# Assignment 4 — Hospital Privacy Dashboard

This notebook walks through the same steps the Streamlit app does, but in small pieces so each part can be checked on its own. The app itself is in `app.py`, and the code here calls the same functions from `database.py`, `auth.py` and `crypto_utils.py`.

Run the cells from top to bottom. The first run makes `hospital.db` and `secret.key` in this folder.

In [1]:
# Step 1: make the tables and the sample data
from database import init_db, get_all_patients

init_db()
patients = get_all_patients()
print(len(patients), "patients")
for p in patients:
    print(p)

6 patients
(1, 'Ahmed Khan', '0300-1234567', 'Flu', None, None, '2026-11-20')
(2, 'Sara Ahmed', '0321-7654321', 'Diabetes', None, None, '2026-11-21')
(3, 'Bilal Hussain', '0333-4567890', 'Fracture', None, None, '2026-11-22')
(4, 'Ayesha Malik', '0345-9876543', 'Asthma', None, None, '2026-11-23')
(5, 'Usman Ali', '0301-2345678', 'Fever', None, None, '2026-11-24')
(6, 'Fatima Noor', '0322-3456789', 'Migraine', None, None, '2026-11-25')


## Login and roles

There are three demo accounts. Passwords are stored as SHA-256 hashes, never as plain text, so even if someone opens the database file they cannot read them. The login still uses the plain passwords from the assignment.

In [2]:
# Step 2: try the three logins, and one wrong password
from auth import check_login, can_do

for user, pw in [("admin", "admin123"), ("Dr. Bob", "doc123"),
                ("Alice_recep", "rec123"), ("admin", "wrong")]:
    result = check_login(user, pw)
    print(user, "->", result["role"] if result else "login failed")

# What each role is allowed to do
for role in ["admin", "doctor", "receptionist"]:
    print(role, "view raw:", can_do(role, "view_raw"),
          "| anonymize:", can_do(role, "anonymize"),
          "| view logs:", can_do(role, "view_logs"))

admin -> admin
Dr. Bob -> doctor
Alice_recep -> receptionist
admin -> login failed
admin view raw: True | anonymize: True | view logs: True
doctor view raw: False | anonymize: False | view logs: False
receptionist view raw: False | anonymize: False | view logs: False


## Masking

A doctor sees anonymized data only, and a receptionist cannot see sensitive data at all. Masking hides the name and the contact number, and the diagnosis is hidden from the receptionist too. The admin sees everything as it is.

In [3]:
# Step 3: masking, and what each role sees for one patient
from crypto_utils import mask_name, mask_contact, show_name, show_contact

name, contact = "Ahmed Khan", "0300-1234567"
print("masked name:", mask_name(name))
print("masked contact:", mask_contact(contact))
print("admin sees:", show_name(name, "ANON_9212", True))
print("doctor sees:", show_name(name, "ANON_9212", False))
print("doctor contact:", show_contact(contact, "XXX-XXX-4567", False))

masked name: A***
masked contact: XXX-XXX-4567
admin sees: Ahmed Khan
doctor sees: ANON_9212
doctor contact: XXX-XXX-4567


## Anonymization

Anonymize replaces each name with an ANON id made from a hash of the name and the patient id, so the same patient always gets the same id. The real name is also kept encrypted with Fernet, so only the admin can read it back. The key lives in `secret.key` and is never pushed to the repo.

In [4]:
# Step 4: anonymize everyone, then read one name back
from database import anonymize_all, get_encrypted_sample
from crypto_utils import decrypt_text, encrypt_text

print("anonymized:", anonymize_all(), "patients")
for p in get_all_patients():
    print(p[0], p[1], "->", p[4], p[5])

name, encrypted = get_encrypted_sample()
print("encrypted:", encrypted)
print("decrypted back:", decrypt_text(encrypted), "| matches:", decrypt_text(encrypted) == name)
print("round trip:", decrypt_text(encrypt_text("test")) == "test")

anonymized: 6 patients
1 Ahmed Khan -> ANON_9212 XXX-XXX-4567
2 Sara Ahmed -> ANON_5879 XXX-XXX-4321
3 Bilal Hussain -> ANON_6674 XXX-XXX-7890
4 Ayesha Malik -> ANON_3748 XXX-XXX-6543
5 Usman Ali -> ANON_1129 XXX-XXX-5678
6 Fatima Noor -> ANON_7870 XXX-XXX-6789
encrypted: gAAAAABrCIOsQFehv_TKzXNY4WtF4NUgS3bIZhffnnhsJ_rmoBV4ErqExqn-oLFcyyiav9-M-PnXYjq1B1v5Q-iTMFrugok0mg==
decrypted back: Ahmed Khan | matches: True
round trip: True


## Add, edit, and the audit log

Every important action is written to the logs table with the user, the role, the time and a short detail. Only the admin can open the logs page. The test patient added below is removed again at the end so the sample data stays as it was.

In [5]:
# Step 5: add a patient, change it, then check the log
from database import add_patient, update_patient, log_action, get_all_logs, get_connection

new_id = add_patient("Test Patient", "0300-9999999", "Cold", "2026-11-26")
log_action(3, "receptionist", "add", f"Added patient Test Patient (ID {new_id})")
update_patient(new_id, "Test Patient", "0300-9999999", "Flu")
log_action(3, "receptionist", "update", f"Updated patient #{new_id}")
print("added patient id:", new_id, "| total now:", len(get_all_patients()))
for row in get_all_logs()[:3]:
    print(row)

# Clean up the test patient again
conn = get_connection()
conn.execute("DELETE FROM patients WHERE patient_id = ?", (new_id,))
conn.commit()
conn.close()
print("after cleanup:", len(get_all_patients()), "patients")

added patient id: 7 | total now: 7
(2, 3, 'receptionist', 'update', '2026-11-27 00:08:28', 'Updated patient #7')
(1, 3, 'receptionist', 'add', '2026-11-27 00:08:28', 'Added patient Test Patient (ID 7)')
after cleanup: 6 patients


## Backup, retention, and activity

The admin can download the patients and the logs as CSV files, which is the backup. Each record also shows how many days old it is, and records older than 365 days are flagged for review, which is the GDPR retention idea. The activity graph in the app counts the log actions per day.

In [6]:
# Step 6: CSV backup and how old the records are
from datetime import date
from database import export_patients_csv, export_logs_csv, get_actions_per_day

print(export_patients_csv().splitlines()[0])
print(export_patients_csv().splitlines()[1])
print(export_logs_csv().splitlines()[0])
for p in get_all_patients():
    days = (date.today() - date.fromisoformat(p[6])).days
    print(p[1], "is", days, "days old")
print("actions per day:", get_actions_per_day())

patient_id,name,contact,diagnosis,anonymized_name,anonymized_contact,date_added
1,Ahmed Khan,0300-1234567,Flu,ANON_9212,XXX-XXX-4567,2026-11-20
log_id,user_id,role,action,timestamp,details
Ahmed Khan is 7 days old
Sara Ahmed is 6 days old
Bilal Hussain is 5 days old
Ayesha Malik is 4 days old
Usman Ali is 3 days old
Fatima Noor is 2 days old
actions per day: [('2026-11-27', 'add', 1), ('2026-11-27', 'update', 1)]


## How the files fit together

The app is split so each file has one job. `database.py` opens SQLite and makes the three tables. `auth.py` checks the login and the roles. `crypto_utils.py` does the masking, the ANON ids and the Fernet encryption. `app.py` is the Streamlit dashboard that ties them together. To run it:

```
pip install -r requirements.txt
streamlit run app.py
```

Demo accounts: Admin `admin` / `admin123`, Doctor `Dr. Bob` / `doc123`, Receptionist `Alice_recep` / `rec123`.